# 02 · Registro de las personas del equipo (CondoHub · IA)

Reconocer a las personas registradas con **embeddings de FaceNet, sin reentrenar la red**, y medir con honestidad:
- se deja **una sesión fuera** a la vez (registro con las demás, prueba con la que quedó fuera),
- el umbral se calibra **solo con impostores** (LFW), no con tus fotos de prueba,
- las fotos donde MTCNN no halla rostro (contraluz, sobreexposición) se **cuentan**, no se esconden.

Estructura en Drive (privada): `MyDrive/condohub-ml/datos/propios/<persona>/<sesion>/foto.jpg` y, por persona, un `consentimiento.json` con `fecha` y `version` (I1).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.environ["ML_DATA_DIR"] = "/content/drive/MyDrive/condohub-ml"
os.makedirs(os.environ["ML_DATA_DIR"], exist_ok=True)

In [ ]:
REPO_URL = ""   # ej. "https://github.com/<org>/condohub-ia.git" (si es privado, usa un token en los Secretos de Colab)
if REPO_URL:
    !git clone --depth 1 {REPO_URL} /content/condohub-ia
else:
    !cp -r "/content/drive/MyDrive/condohub-ml/codigo/condohub-ia" /content/condohub-ia
%cd /content/condohub-ia
!pip install -q facenet-pytorch --no-deps
!pip install -q scikit-learn pyyaml matplotlib pillow

In [ ]:
import json, yaml, numpy as np
from pathlib import Path
from ml.src import embeddings, lfw, propios, registro
from ml.src.paths import dir_datos, dir_experimento

cfg = yaml.safe_load(open("ml/configs/propios_registro.yaml", encoding="utf-8"))
fotos = propios.listar_fotos(dir_datos() / cfg["carpeta_propios"])   # falla si falta algún consentimiento.json (I1)
from collections import Counter
print(Counter((f.persona, f.sesion) for f in fotos))

In [ ]:
embeber = embeddings.crear_embebedor()
imagenes = [propios.cargar_imagen(f.ruta) for f in fotos]
emb, detectado = embeber(imagenes)
personas = np.array([f.persona for f in fotos]); sesiones = np.array([f.sesion for f in fotos])

# Tasa de detección por sesión: cuántas fotos pierde MTCNN (contraluz, sobreexposición...)
deteccion = {f"{p}/{s}": f"{int(detectado[(personas == p) & (sesiones == s)].sum())}/{int(((personas == p) & (sesiones == s)).sum())}" for p, s in sorted(set(zip(personas, sesiones)))}
print(deteccion)
emb, personas, sesiones = emb[detectado], personas[detectado], sesiones[detectado]   # solo las detectadas entran al registro y a la prueba

In [ ]:
# Desconocidos: personas de LFW (público). Son impostores: no deben ser reconocidas.
img_lfw, et_lfw, _ = lfw.cargar_lfw(cfg["min_fotos_lfw"])
emb_lfw, det_lfw = embeber(img_lfw)
emb_lfw, et_lfw = emb_lfw[det_lfw], et_lfw[det_lfw]
print(f"{len(np.unique(et_lfw))} impostores, {len(et_lfw)} fotos")

In [ ]:
resultado = registro.evaluar(emb, personas, sesiones, emb_lfw, et_lfw, cfg)
resultado["deteccion_por_sesion"] = deteccion
destino = dir_experimento(cfg["nombre"])
(destino / "metrics.json").write_text(json.dumps(resultado, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps({k: resultado[k] for k in ("umbral_calibrado_con_impostores_de_validacion", "far_prueba", "frr", "confusiones")}, indent=2))
for fila in resultado["por_sesion"]: print(fila)
print("guardado en", destino)

## Cómo leer el resultado

- **far_prueba**: fracción de desconocidos de LFW que el sistema habría aceptado. Debe quedar cerca de `far_maximo`.
- **frr**: fracción de tus fotos (de la sesión dejada fuera) que no se reconocen. Mira `por_sesion` para ver si la luz baja o alta es la que falla.
- **confusiones**: fotos reconocidas como OTRA persona registrada (solo existen cuando hay 2 o más personas).
- **deteccion_por_sesion**: las fotos que MTCNN no encontró; si son muchas en luz alta, el problema está antes del reconocimiento.

Con una sola persona el FRR sale de pocas sesiones: tómalo como una primera señal, no como una cifra definitiva. Las métricas copiadas al repo deben ir **sin fotos** ni salidas con imágenes.